In [ ]:
#!/usr/bin/env python3

import json
from smolagents import tool, InferenceClientModel
from smolagents.agents import ToolCallingAgent, CodeAgent
from smolagents.models import ChatMessage, MessageRole, TokenUsage, ChatMessageToolCall, ChatMessageToolCallFunction, Model
from smolagents.default_tools import WebSearchTool

# Create a simple mock model for testing

# class Model:
#     def __init__(self):
#         self.model_id = "mock-model"
#         self.call_count = 0
    
#     def generate(self, messages, **kwargs):
#         self.call_count += 1
        
#         # Simulate structured output
#         if kwargs.get("response_format"):
#             content = json.dumps({
#                 "thought": f"This is my reasoning for step {self.call_count}",
#                 "tool_calls": [
#                     {
#                         "name": "final_answer",
#                         "arguments": {"answer": "42"}
#                     }
#                 ]
#             })
#         else:
#             content = "Regular model output"
        
#         return ChatMessage(
#             role=MessageRole.ASSISTANT,
#             content=content,
#             token_usage=TokenUsage(input_tokens=10, output_tokens=20),
#         )
    
#     def to_dict(self):
#         return {"model_id": self.model_id}
    
#     @classmethod
#     def from_dict(cls, data):
#         return cls()


@tool
def dummy_tool(query: str) -> str:
    """A dummy tool for testing
    
    Args:
        query: A query string
    """
    return f"Result: {query}"


def test_basic_initialization():
    """Test that the agent initializes with the new parameter"""
    print("Test 1: Basic initialization...")
    model = InferenceClientModel()
    agent = ToolCallingAgent(
        tools=[dummy_tool],
        model=model,
        use_structured_outputs_internally=True,
    )
    assert agent._use_structured_outputs_internally is True
    print("✓ Agent initializes correctly with structured outputs enabled")


def test_without_structured_outputs():
    """Test that the agent still works without structured outputs"""
    print("\nTest 2: Without structured outputs...")
    model = InferenceClientModel()
    agent = ToolCallingAgent(
        tools=[dummy_tool],
        model=model,
        use_structured_outputs_internally=False,
    )
    assert agent._use_structured_outputs_internally is False
    print("✓ Agent initializes correctly without structured outputs")


def test_serialization():
    """Test that to_dict and from_dict work"""
    print("\nTest 3: Serialization...")
    model = InferenceClientModel()
    agent = ToolCallingAgent(
        tools=[dummy_tool],
        model=model,
        use_structured_outputs_internally=True,
    )
    
    # Test to_dict
    agent_dict = agent.to_dict()
    assert "use_structured_outputs_internally" in agent_dict
    assert agent_dict["use_structured_outputs_internally"] is True
    print("✓ to_dict includes the structured outputs flag")
    
    # Test from_dict
    restored_agent = ToolCallingAgent.from_dict(agent_dict)
    assert restored_agent._use_structured_outputs_internally is True
    print("✓ from_dict restores the structured outputs flag")


def test_prompt_template_loading():
    """Test that the correct prompt template is loaded"""
    print("\nTest 4: Prompt template loading...")
    model = InferenceClientModel()
    
    # With structured outputs
    agent_structured = ToolCallingAgent(
        tools=[dummy_tool],
        model=model,
        use_structured_outputs_internally=True,
    )
    assert agent_structured.prompt_templates is not None
    assert "JSON" in agent_structured.system_prompt or "json" in agent_structured.system_prompt
    print("✓ Structured prompts loaded correctly")
    
    # Without structured outputs
    agent_normal = ToolCallingAgent(
        tools=[dummy_tool],
        model=model,
        use_structured_outputs_internally=False,
    )
    assert agent_normal.prompt_templates is not None
    print("✓ Normal prompts loaded correctly")


def test_run_with_reasoning():
    """Test a full run with reasoning capture"""
    print("\nTest 5: Full run with reasoning...")
    model = InferenceClientModel()
    agent = ToolCallingAgent(
        tools=[dummy_tool],
        model=model,
        use_structured_outputs_internally=True,
        max_steps=1,
    )
    
    try:
        result = agent.run("What is 2+2?", return_full_result=True)
        
        # Check that steps were recorded
        assert len(result.steps) > 0
        print(f"✓ Agent completed run with {len(result.steps)} steps")
        
        # Check if thought was captured
        action_steps = [s for s in result.steps if s.get("type") == "ActionStep"]
        if action_steps and action_steps[0].get("thought"):
            print(f"✓ Thought captured: '{action_steps[0]['thought']}'")
        else:
            print("⚠ Warning: Thought was not captured (might be expected depending on model behavior)")
            
    except Exception as e:
        print(f"⚠ Run test had an issue (might be expected with mock): {e}")

def test_run_without_reasoning():
    """Test a full run without reasoning capture"""
    print("\nTest 5: Full run without reasoning...")
    model = InferenceClientModel()
    agent = ToolCallingAgent(
        tools=[dummy_tool],
        model=model,
        # use_structured_outputs_internally=False,
        max_steps=2,
    )
    
    try:
        result = agent.run("What is 2+2?", return_full_result=True)
        
        # Check that steps were recorded
        assert len(result.steps) > 0
        print(f"✓ Agent completed run with {len(result.steps)} steps")
        
        # Check if thought was captured
        action_steps = [s for s in result.steps if s.get("type") == "ActionStep"]
        if action_steps and action_steps[0].get("thought"):
            print(f"✓ Thought captured: '{action_steps[0]['thought']}'")
        else:
            print("⚠ Warning: Thought was not captured (might be expected depending on model behavior)")
            
    except Exception as e:
        print(f"⚠ Run test had an issue (might be expected with mock): {e}")


In [ ]:
print("=" * 60)
print("Testing ToolCallingAgent Reasoning Feature")
print("=" * 60)


# test_basic_initialization()
# test_without_structured_outputs()
# test_serialization()
# test_prompt_template_loading()
# test_run_with_reasoning()
test_run_without_reasoning()

print("\n" + "=" * 60)
print("All tests passed! ✓")
print("=" * 60)


    def __init__(
        self,
        tools: list[Tool],
        model: Model,
        prompt_templates: PromptTemplates | None = None,
        planning_interval: int | None = None,
        stream_outputs: bool = False,
        max_tool_threads: int | None = None,
        use_structured_outputs_internally: bool = False,
        **kwargs,
    ):

In [3]:
import json
from smolagents import tool, InferenceClientModel
from smolagents.agents import ToolCallingAgent, CodeAgent
from smolagents.models import ChatMessage, MessageRole, TokenUsage, ChatMessageToolCall, ChatMessageToolCallFunction, Model
from smolagents.default_tools import WebSearchTool

model = InferenceClientModel()
agent = ToolCallingAgent(tools=[WebSearchTool()], model=model, stream_outputs=True, use_structured_outputs_internally=True,)

print (agent)
agent.run("How many seconds would it take for a leopard at full speed to run through Pont des Arts?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many seconds would it take for a leopard at full speed to run through Pont des Arts?                        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

c:\Users\Ayush\Documents\good_first_issues\smolagentshf\smolenv\Lib\site-packages\rich\live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────────── Agent Reasoning ────────────────────────────────────────────────╮
│ To determine how many seconds it would take for a leopard to run through Pont des Arts (PDA), I need to know    │
│ two things: the length of PDA and the full running speed of a leopard. I'll start by searching for the length   │
│ of Pont des Arts.                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'length of Pont des Arts'}                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Step 1: Duration 6.09 seconds| Input tokens: 1,295 | Output tokens: 97]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
'str' object has no attribute 'role'

[Step 2: Duration 0.00 seconds]

AgentGenerationError: Error while generating output:
'str' object has no attribute 'role'

In [27]:
import json
from smolagents import tool, InferenceClientModel
from smolagents.agents import ToolCallingAgent, CodeAgent
from smolagents.models import ChatMessage, MessageRole, TokenUsage, ChatMessageToolCall, ChatMessageToolCallFunction, Model
from smolagents.default_tools import WebSearchTool

from smolagents import ToolCallingAgent, Tool, InferenceClientModel
from smolagents import ChatMessage


# Define a simple tool
class AddNumbersTool(Tool):
    name = "add_numbers"
    description = "Adds two numbers together."
    inputs = {
        "a": {"type": "number", "description": "The first number"},
        "b": {"type": "number", "description": "The second number"},
    }
    output_type = "number"

    def forward(self, a: float, b: float):  # <-- use forward, not __call__
        return a + b


# Initialize model
model = InferenceClientModel()

# Create the ToolCallingAgent
agent = ToolCallingAgent(
    tools=[AddNumbersTool()],
    model=model,
    stream_outputs=False,
    use_structured_outputs_internally=True,
    max_steps=3,
)

# Run a simple query
result = agent.run("Add 42 and 58, and tell me the result in a complete sentence.")


print("\nFinal Answer:")
print(result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 42 and 58, and tell me the result in a complete sentence.                                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭──────────────────────────────────────────────── Agent Reasoning ────────────────────────────────────────────────╮
│ I need to add the two numbers 42 and 58 to get their sum. After obtaining the result, I'll provide the answer   │
│ in a complete sentence.                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'add_numbers' with arguments: {'a': 42, 'b': 58}                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 100

[Step 1: Duration 6.22 seconds| Input tokens: 1,287 | Output tokens: 80]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
'str' object has no attribute 'role'

[Step 2: Duration 0.01 seconds]

AgentGenerationError: Error while generating output:
'str' object has no attribute 'role'

In [ ]:
from smolagents.agents import ToolCallingAgent
from smolagents import Tool
from smolagents import ChatMessage
from smolagents.models import InferenceClientModel

class AddNumbersTool(Tool):
    name = "add_numbers"
    description = "Adds two numbers together."
    inputs = {
        "a": {"type": "number", "description": "First number."},
        "b": {"type": "number", "description": "Second number."},
    }
    output_type = "number"
    def forward(self, a, b):
        return a + b


# Model
model = InferenceClientModel()

# Agent
agent = ToolCallingAgent(
    tools=[AddNumbersTool()],  # must be instance, not class
    model=model,
    stream_outputs=False,
    use_structured_outputs_internally=True,
)

# Run
result = agent.run("subtract 42 and 58, and tell me the result in a complete sentence.")

print(result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ subtract 42 and 58, and tell me the result in a complete sentence.                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭──────────────────────────────────────────────── Agent Reasoning ────────────────────────────────────────────────╮
│ To subtract 42 from 58, I will use the add_numbers tool, setting one of the numbers to negative. This will      │
│ effectively perform 58 - 42.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'add_numbers' with arguments: {'a': 58, 'b': -42}                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 16

[Step 1: Duration 3.93 seconds| Input tokens: 1,282 | Output tokens: 84]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
'str' object has no attribute 'role'

[Step 2: Duration 0.00 seconds]

AgentGenerationError: Error while generating output:
'str' object has no attribute 'role'